> Experiments with [data](../trapallada/data.py)  module

In [1]:
import pathlib

import yaml

import jax.random as jr

import trapallada.data
import trapallada.model.nn
import trapallada.util
import trapallada.core

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


A parameters file that specifies an experiment

In [3]:
# parameters_file = pathlib.Path.cwd().parent / 'parameters.yaml'
parameters_file = trapallada.util.get_base_directory() / 'parameters.yaml'
assert parameters_file.exists()

It is read into a `dic`

In [4]:
np.random.RandomState(42).choice([1, 2, 3], size=10) 

array([3, 1, 3, 3, 1, 1, 3, 2, 3, 3])

In [5]:
with parameters_file.open() as f:
    parameters = yaml.safe_load(f)

In [6]:
%pycat {parameters_file}

PRNG seeds:
  model initialization:
    decoder: 43
    encoder: 42
  sampling:
    training set: 14
    validation set: 15
  training dataloader: 43
  training-validation splitting: 42
loss:
  x_sd: 0.15 # <---- FFT
  beta: 1.
model: &model
  channels:
  - 12
  - 12
  - 12
  dim_z: 40
  kernel_sizes: # <---- FFT (due to redundancy)
  - 64
  - 32
  - 16
  # stride: 2
  stride:
  - 2
  - 2
  - 2
data:
    fft: false # <---- FFT
training:
  batch_size: 32
  dl_rng_key: 43
  learning_rate: 3.0e-05
  n_epochs: 2
  
  # ratio of the dataset to use for validation
  train_test_split:
    test_size: 0.2
  
  # whether or not to ignore non "good time intervals" (GTIs) in the loss function; it only
  # makes sense when working with time-domain signals, and will be ignored for frequency-domain ones
  mask_non_gti_intervals: true
VAE:
  class: VAE
  encoder:
    class: Encoder
    parameters:
      << : [*model]
  decoder:
    class: Decoder # <---- FFT (NonNegativeDecoder/NonNegativeConditionalDe

# sample_dataset_parameters

It returns the files making up a *sample* dataset (that are expected by `trapallada.data.SignalsDataset`):


- data itself, in the form of an `Array`

- mask, another `Array`

- metadata in a *Pandas* `DataFrame`

In [7]:
trapallada.data.sample_dataset_parameters()

{'signals_filename': PosixPath('/home/manu/Sync/git/trapallada/data/rates_large_infs/signals.pt'),
 'masks_filename': PosixPath('/home/manu/Sync/git/trapallada/data/rates_large_infs/masks.pt'),
 'metadata_filename': PosixPath('/home/manu/Sync/git/trapallada/data/rates_large_infs/metadata.parquet')}

It is turned into a *PyTorch* `Dataset`. By default, *windows* are **not** normalized.

In [8]:
# ds = trapallada.data.SignalsDataset(signals_file, metadata_file, masks_file, fft=True)
ds = trapallada.data.SignalsDataset(**trapallada.data.sample_dataset_parameters(), fft=parameters['data']['fft'])
len(ds), ds[0][1].shape

(30856, torch.Size([5, 225]))

Inside the `Dataset`, there are `array`s

In [9]:
ds.signals.shape, ds.masks.shape

(torch.Size([30856, 5, 225]), (30856, 225))

In [10]:
has_label = ds.metadata_df['source'].isin([e + '.FTZ' for e in trapallada.data.Labels().signals_without_prefix])
i_to_validation = np.flatnonzero(has_label)

# dataloaders_from_dataset

`DataLoaders` are derived from the `Dataset`

In [11]:
train_dl, validation_dl = trapallada.data.dataloaders_from_dataset(
    ds,
    parameters['training']['train_test_split'] | dict(random_state=parameters['PRNG seeds']['training-validation splitting']),
    parameters['training']['batch_size'], parameters['PRNG seeds']['training dataloader'], i_to_validation=i_to_validation
)


How many samples per batch?

In [12]:
batch = next(iter(train_dl))
len(batch)

3

In the first batch
- the *index component* 

In [13]:
batch[0]

tensor([26916, 14627, 15126,  7868, 20888, 22583,  8644, 24856, 14128, 13142,
        26300, 24189, 24886,   961, 16928, 26284,  7914, 20962, 25204,  1657,
        28508, 30429, 11752, 23807, 10551, 18139,  4775, 16562, 15610,  1653,
          100, 18255])

- the signals

In [14]:
batch[1].shape

torch.Size([32, 5, 225])

- the masks

In [15]:
batch[2].shape

torch.Size([32, 225])

- the labels (or `None`, if unsupervised)

The input size is expected along with the parameters

In [16]:
parameters['VAE']['encoder']['parameters']['input_size'] = trapallada.util.input_size_from_dataset(ds)

In [17]:
x = batch[1][0].numpy()
encoder = trapallada.model.nn.Encoder(
    **parameters['VAE']['encoder']['parameters'],
    key=jr.PRNGKey(parameters['PRNG seeds']['model initialization']['encoder']))
encoder.debug_call(x)
print(encoder)

initial shape: (5, 225)
after equinox.nn._conv: (12, 81)
after jax._src.nn.functions: (12, 81)
after equinox.nn._normalisation: (12, 81)
after equinox.nn._conv: (12, 25)
after jax._src.nn.functions: (12, 25)
after equinox.nn._normalisation: (12, 25)
after equinox.nn._conv: (12, 5)
after jax._src.nn.functions: (12, 5)
after equinox.nn._normalisation: (12, 5)
after jax.numpy: (60,)
after equinox.nn._linear: (80,)
after trapallada.model.nn: (80,)
Encoder(
  layers=[
    Conv1d(
      num_spatial_dims=1,
      weight=f32[12,5,64],
      bias=f32[12,1],
      in_channels=5,
      out_channels=12,
      kernel_size=(64,),
      stride=(2,),
      padding=((0, 0),),
      dilation=(1,),
      groups=1,
      use_bias=True,
      padding_mode='ZEROS'
    ),
    <PjitFunction of <function relu at 0x7f1e47e51d00>>,
    LayerNorm(
      shape=(12, 81),
      eps=1e-05,
      use_weight=True,
      use_bias=True,
      weight=f32[12,81],
      bias=f32[12,81]
    ),
    Conv1d(
      num_spatial_d

# Labels

A class to access the (scarce) available *labels*.

In [18]:
labels = trapallada.data.Labels()
labels

Reference signals:
                                   signal alias
0  0605950101/P0605950101PNX000SRCTSR8001  Javi
1  0690744401/P0690744401PNX000SRCTSR8002     I
2  0740290301/P0740290301PNX000SRCTSR8002   III
3  0720173901/P0720173901PNX000SRCTSR8001    IV

In [19]:
labels.signals, labels.reference_signals_and_labels

(['0605950101/P0605950101PNX000SRCTSR8001',
  '0690744401/P0690744401PNX000SRCTSR8002',
  '0740290301/P0740290301PNX000SRCTSR8002',
  '0720173901/P0720173901PNX000SRCTSR8001'],
 [['0605950101/P0605950101PNX000SRCTSR8001', 'Javi'],
  ['0690744401/P0690744401PNX000SRCTSR8002', 'I'],
  ['0740290301/P0740290301PNX000SRCTSR8002', 'III'],
  ['0720173901/P0720173901PNX000SRCTSR8001', 'IV']])

In [20]:
trapallada.data.labels_files()

{'reference_signals_file': PosixPath('/home/manu/Sync/git/trapallada/interesting.yaml'),
 'annotated_windows_file': PosixPath('/home/manu/Sync/git/trapallada/annotations/annotated_windows.parquet')}

In [21]:
labels = trapallada.data.Labels(**trapallada.data.labels_files())
labels

Reference signals:
                                   signal alias
0  0605950101/P0605950101PNX000SRCTSR8001  Javi
1  0690744401/P0690744401PNX000SRCTSR8002     I
2  0740290301/P0740290301PNX000SRCTSR8002   III
3  0720173901/P0720173901PNX000SRCTSR8001    IV

In [22]:
print(len(labels.signals_without_prefix))
labels.signals_without_prefix[:10]

194


['P0605950101PNX000SRCTSR8001',
 'P0690744401PNX000SRCTSR8002',
 'P0740290301PNX000SRCTSR8002',
 'P0720173901PNX000SRCTSR8001',
 'P0201360201PNX000SRCTSR8002',
 'P0770180201PNX000SRCTSR8001',
 'P0652330101PNX000SRCTSR8001',
 'P0764800301PNX000SRCTSR8005',
 'P0761970101PNX000SRCTSR8002',
 'P0723700101PNX000SRCTSR8002']

No duplicates

In [23]:
assert len(np.unique(labels.signals_without_prefix)) == len(labels.signals_without_prefix)

In [24]:
labels.reference_signals_and_labels

[['0605950101/P0605950101PNX000SRCTSR8001', 'Javi'],
 ['0690744401/P0690744401PNX000SRCTSR8002', 'I'],
 ['0740290301/P0740290301PNX000SRCTSR8002', 'III'],
 ['0720173901/P0720173901PNX000SRCTSR8001', 'IV']]